# Text generation with an RNN

## Setup

### Import TensorFlow and other libraries

In [2]:
import tensorflow as tf

import numpy as np
import os
import time

### Dataset

Change the following line to run this code on your own data.

In [3]:
#from google.colab import files
#uploaded = files.upload()

Saving 374451545329457.txt to 374451545329457.txt


### Read the data

First, look in the text:

In [4]:
# Read, then decode for py2 compat.
text = open("374451545329457.txt", 'rb').read().decode(encoding='utf-8')
text = text.replace("|","",text.count("|"))
# length of text is the number of characters in it
print(f'Length of text: {len(text)} characters')

Length of text: 2554631 characters


In [5]:
# Take a look at the first 250 characters in text
print(text[:250])

به نام خداوند جان و خرد
کزین برتر اندیشه برنگذرد
خداوند نام و خداوند جای
خداوند روزی ده رهنمای
خداوند کیوان و گردان سپهر
فروزنده ماه و ناهید و مهر
ز نام و نشان و گمان برترست
نگارندهٔ بر شده پیکرست
به بینندگان آفریننده را
نبینی مرنجان دو بیننده را
نیا


In [6]:
# The unique characters in the file
vocab = sorted(set(text))
print(f'{len(vocab)} unique characters')

47 unique characters


## Process the text

### Vectorize the text

Before training, you need to convert the strings to a numerical representation.

The `tf.keras.layers.StringLookup` layer can convert each character into a numeric ID. It just needs the text to be split into tokens first.

In [7]:
example_texts = ['abcdefg', 'xyz']

chars = tf.strings.unicode_split(example_texts, input_encoding='UTF-8')
chars

<tf.RaggedTensor [[b'a', b'b', b'c', b'd', b'e', b'f', b'g'], [b'x', b'y', b'z']]>

Now create the `tf.keras.layers.StringLookup` layer:

In [8]:
ids_from_chars = tf.keras.layers.StringLookup(
    vocabulary=list(vocab), mask_token=None)

It converts from tokens to character IDs:

In [9]:
ids = ids_from_chars(chars)
ids

<tf.RaggedTensor [[0, 0, 0, 0, 0, 0, 0], [0, 0, 0]]>

Since the goal of this tutorial is to generate text, it will also be important to invert this representation and recover human-readable strings from it. For this you can use `tf.keras.layers.StringLookup(..., invert=True)`.  

Note: Here instead of passing the original vocabulary generated with `sorted(set(text))` use the `get_vocabulary()` method of the `tf.keras.layers.StringLookup` layer so that the `[UNK]` tokens is set the same way.

In [10]:
chars_from_ids = tf.keras.layers.StringLookup(
    vocabulary=ids_from_chars.get_vocabulary(), invert=True, mask_token=None)

This layer recovers the characters from the vectors of IDs, and returns them as a `tf.RaggedTensor` of characters:

In [11]:
chars = chars_from_ids(ids)
chars

<tf.RaggedTensor [[b'[UNK]', b'[UNK]', b'[UNK]', b'[UNK]', b'[UNK]', b'[UNK]', b'[UNK]'],
 [b'[UNK]', b'[UNK]', b'[UNK]']]>

You can `tf.strings.reduce_join` to join the characters back into strings.

In [12]:
tf.strings.reduce_join(chars, axis=-1).numpy()

array([b'[UNK][UNK][UNK][UNK][UNK][UNK][UNK]', b'[UNK][UNK][UNK]'],
      dtype=object)

In [13]:
def text_from_ids(ids):
  return tf.strings.reduce_join(chars_from_ids(ids), axis=-1)

### The prediction task

Given a character, or a sequence of characters, what is the most probable next character? This is the task you're training the model to perform. The input to the model will be a sequence of characters, and you train the model to predict the output—the following character at each time step.

Since RNNs maintain an internal state that depends on the previously seen elements, given all the characters computed until this moment, what is the next character?


### Create training examples and targets

Next divide the text into example sequences. Each input sequence will contain `seq_length` characters from the text.

For each input sequence, the corresponding targets contain the same length of text, except shifted one character to the right.

So break the text into chunks of `seq_length+1`. For example, say `seq_length` is 4 and our text is "Hello". The input sequence would be "Hell", and the target sequence "ello".

To do this first use the `tf.data.Dataset.from_tensor_slices` function to convert the text vector into a stream of character indices.

In [14]:
all_ids = ids_from_chars(tf.strings.unicode_split(text, 'UTF-8'))
all_ids

<tf.Tensor: shape=(2554631,), dtype=int64, numpy=array([15, 38,  2, ..., 46, 37,  1])>

In [15]:
ids_dataset = tf.data.Dataset.from_tensor_slices(all_ids)

In [16]:
for ids in ids_dataset.take(10):
    print(chars_from_ids(ids).numpy().decode('utf-8'))

ب
ه
 
ن
ا
م
 
خ
د
ا


In [17]:
seq_length = 100


The `batch` method lets you easily convert these individual characters to sequences of the desired size.

In [18]:
sequences = ids_dataset.batch(seq_length+1, drop_remainder=True)

for seq in sequences.take(1):
  print(chars_from_ids(seq))

tf.Tensor(
[b'\xd8\xa8' b'\xd9\x87' b' ' b'\xd9\x86' b'\xd8\xa7' b'\xd9\x85' b' '
 b'\xd8\xae' b'\xd8\xaf' b'\xd8\xa7' b'\xd9\x88' b'\xd9\x86' b'\xd8\xaf'
 b' ' b'\xd8\xac' b'\xd8\xa7' b'\xd9\x86' b' ' b'\xd9\x88' b' '
 b'\xd8\xae' b'\xd8\xb1' b'\xd8\xaf' b'\n' b'\xda\xa9' b'\xd8\xb2'
 b'\xdb\x8c' b'\xd9\x86' b' ' b'\xd8\xa8' b'\xd8\xb1' b'\xd8\xaa'
 b'\xd8\xb1' b' ' b'\xd8\xa7' b'\xd9\x86' b'\xd8\xaf' b'\xdb\x8c'
 b'\xd8\xb4' b'\xd9\x87' b' ' b'\xd8\xa8' b'\xd8\xb1' b'\xd9\x86'
 b'\xda\xaf' b'\xd8\xb0' b'\xd8\xb1' b'\xd8\xaf' b'\n' b'\xd8\xae'
 b'\xd8\xaf' b'\xd8\xa7' b'\xd9\x88' b'\xd9\x86' b'\xd8\xaf' b' '
 b'\xd9\x86' b'\xd8\xa7' b'\xd9\x85' b' ' b'\xd9\x88' b' ' b'\xd8\xae'
 b'\xd8\xaf' b'\xd8\xa7' b'\xd9\x88' b'\xd9\x86' b'\xd8\xaf' b' '
 b'\xd8\xac' b'\xd8\xa7' b'\xdb\x8c' b'\n' b'\xd8\xae' b'\xd8\xaf'
 b'\xd8\xa7' b'\xd9\x88' b'\xd9\x86' b'\xd8\xaf' b' ' b'\xd8\xb1'
 b'\xd9\x88' b'\xd8\xb2' b'\xdb\x8c' b' ' b'\xd8\xaf' b'\xd9\x87' b' '
 b'\xd8\xb1' b'\xd9\x87' b'\xd9\x86' b'\xd

It's easier to see what this is doing if you join the tokens back into strings:

In [19]:
for seq in sequences.take(5):
  print(text_from_ids(seq).numpy())

b'\xd8\xa8\xd9\x87 \xd9\x86\xd8\xa7\xd9\x85 \xd8\xae\xd8\xaf\xd8\xa7\xd9\x88\xd9\x86\xd8\xaf \xd8\xac\xd8\xa7\xd9\x86 \xd9\x88 \xd8\xae\xd8\xb1\xd8\xaf\n\xda\xa9\xd8\xb2\xdb\x8c\xd9\x86 \xd8\xa8\xd8\xb1\xd8\xaa\xd8\xb1 \xd8\xa7\xd9\x86\xd8\xaf\xdb\x8c\xd8\xb4\xd9\x87 \xd8\xa8\xd8\xb1\xd9\x86\xda\xaf\xd8\xb0\xd8\xb1\xd8\xaf\n\xd8\xae\xd8\xaf\xd8\xa7\xd9\x88\xd9\x86\xd8\xaf \xd9\x86\xd8\xa7\xd9\x85 \xd9\x88 \xd8\xae\xd8\xaf\xd8\xa7\xd9\x88\xd9\x86\xd8\xaf \xd8\xac\xd8\xa7\xdb\x8c\n\xd8\xae\xd8\xaf\xd8\xa7\xd9\x88\xd9\x86\xd8\xaf \xd8\xb1\xd9\x88\xd8\xb2\xdb\x8c \xd8\xaf\xd9\x87 \xd8\xb1\xd9\x87\xd9\x86\xd9\x85\xd8\xa7\xdb\x8c\n\xd8\xae\xd8\xaf\xd8\xa7\xd9\x88\xd9\x86\xd8\xaf'
b' \xda\xa9\xdb\x8c\xd9\x88\xd8\xa7\xd9\x86 \xd9\x88 \xda\xaf\xd8\xb1\xd8\xaf\xd8\xa7\xd9\x86 \xd8\xb3\xd9\xbe\xd9\x87\xd8\xb1\n\xd9\x81\xd8\xb1\xd9\x88\xd8\xb2\xd9\x86\xd8\xaf\xd9\x87 \xd9\x85\xd8\xa7\xd9\x87 \xd9\x88 \xd9\x86\xd8\xa7\xd9\x87\xdb\x8c\xd8\xaf \xd9\x88 \xd9\x85\xd9\x87\xd8\xb1\n\xd8\xb2 \xd9\x86\xd8\

For training you'll need a dataset of `(input, label)` pairs. Where `input` and
`label` are sequences. At each time step the input is the current character and the label is the next character.

Here's a function that takes a sequence as input, duplicates, and shifts it to align the input and label for each timestep:

In [20]:
def split_input_target(sequence):
    input_text = sequence[:-1]
    target_text = sequence[1:]
    return input_text, target_text

In [21]:
split_input_target(list("Tensorflow"))

(['T', 'e', 'n', 's', 'o', 'r', 'f', 'l', 'o'],
 ['e', 'n', 's', 'o', 'r', 'f', 'l', 'o', 'w'])

In [22]:
dataset = sequences.map(split_input_target)

In [23]:
for input_example, target_example in dataset.take(1):
    print("Input :", text_from_ids(input_example).numpy())
    print("Target:", text_from_ids(target_example).numpy())

Input : b'\xd8\xa8\xd9\x87 \xd9\x86\xd8\xa7\xd9\x85 \xd8\xae\xd8\xaf\xd8\xa7\xd9\x88\xd9\x86\xd8\xaf \xd8\xac\xd8\xa7\xd9\x86 \xd9\x88 \xd8\xae\xd8\xb1\xd8\xaf\n\xda\xa9\xd8\xb2\xdb\x8c\xd9\x86 \xd8\xa8\xd8\xb1\xd8\xaa\xd8\xb1 \xd8\xa7\xd9\x86\xd8\xaf\xdb\x8c\xd8\xb4\xd9\x87 \xd8\xa8\xd8\xb1\xd9\x86\xda\xaf\xd8\xb0\xd8\xb1\xd8\xaf\n\xd8\xae\xd8\xaf\xd8\xa7\xd9\x88\xd9\x86\xd8\xaf \xd9\x86\xd8\xa7\xd9\x85 \xd9\x88 \xd8\xae\xd8\xaf\xd8\xa7\xd9\x88\xd9\x86\xd8\xaf \xd8\xac\xd8\xa7\xdb\x8c\n\xd8\xae\xd8\xaf\xd8\xa7\xd9\x88\xd9\x86\xd8\xaf \xd8\xb1\xd9\x88\xd8\xb2\xdb\x8c \xd8\xaf\xd9\x87 \xd8\xb1\xd9\x87\xd9\x86\xd9\x85\xd8\xa7\xdb\x8c\n\xd8\xae\xd8\xaf\xd8\xa7\xd9\x88\xd9\x86'
Target: b'\xd9\x87 \xd9\x86\xd8\xa7\xd9\x85 \xd8\xae\xd8\xaf\xd8\xa7\xd9\x88\xd9\x86\xd8\xaf \xd8\xac\xd8\xa7\xd9\x86 \xd9\x88 \xd8\xae\xd8\xb1\xd8\xaf\n\xda\xa9\xd8\xb2\xdb\x8c\xd9\x86 \xd8\xa8\xd8\xb1\xd8\xaa\xd8\xb1 \xd8\xa7\xd9\x86\xd8\xaf\xdb\x8c\xd8\xb4\xd9\x87 \xd8\xa8\xd8\xb1\xd9\x86\xda\xaf\xd8\xb0\xd8\xb1\

### Create training batches

You used `tf.data` to split the text into manageable sequences. But before feeding this data into the model, you need to shuffle the data and pack it into batches.

In [24]:
# Batch size
BATCH_SIZE = 64

# Buffer size to shuffle the dataset
# (TF data is designed to work with possibly infinite sequences,
# so it doesn't attempt to shuffle the entire sequence in memory. Instead,
# it maintains a buffer in which it shuffles elements).
BUFFER_SIZE = 10000

dataset = (
    dataset
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(tf.data.experimental.AUTOTUNE))

dataset

<_PrefetchDataset element_spec=(TensorSpec(shape=(64, 100), dtype=tf.int64, name=None), TensorSpec(shape=(64, 100), dtype=tf.int64, name=None))>

## Build The Model

This section defines the model as a `keras.Model` subclass (For details see [Making new Layers and Models via subclassing](https://www.tensorflow.org/guide/keras/custom_layers_and_models)).

This model has three layers:

* `tf.keras.layers.Embedding`: The input layer. A trainable lookup table that will map each character-ID to a vector with `embedding_dim` dimensions;
* `tf.keras.layers.GRU`: A type of RNN with size `units=rnn_units` (You can also use an LSTM layer here.)
* `tf.keras.layers.Dense`: The output layer, with `vocab_size` outputs. It outputs one logit for each character in the vocabulary. These are the log-likelihood of each character according to the model.

In [25]:
# Length of the vocabulary in StringLookup Layer
vocab_size = len(ids_from_chars.get_vocabulary())

# The embedding dimension
embedding_dim = 256

# Number of RNN units
rnn_units = 1024

In [26]:
class MyModel(tf.keras.Model):
  def __init__(self, vocab_size, embedding_dim, rnn_units):
    super().__init__(self)
    self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
    self.gru = tf.keras.layers.GRU(rnn_units,
                                   return_sequences=True,
                                   return_state=True)
    self.dense = tf.keras.layers.Dense(vocab_size)

  def call(self, inputs, states=None, return_state=False, training=False):
    x = inputs
    x = self.embedding(x, training=training)
    if states is None:
      states = self.gru.get_initial_state(x)
    x, states = self.gru(x, initial_state=states, training=training)
    x = self.dense(x, training=training)

    if return_state:
      return x, states
    else:
      return x

In [27]:
model = MyModel(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    rnn_units=rnn_units)

For each character the model looks up the embedding, runs the GRU one timestep with the embedding as input, and applies the dense layer to generate logits predicting the log-likelihood of the next character:

![A drawing of the data passing through the model](https://github.com/tensorflow/text/blob/master/docs/tutorials/images/text_generation_training.png?raw=1)

Note: For training you could use a `keras.Sequential` model here. To  generate text later you'll need to manage the RNN's internal state. It's simpler to include the state input and output options upfront, than it is to rearrange the model architecture later. For more details see the [Keras RNN guide](https://www.tensorflow.org/guide/keras/rnn#rnn_state_reuse).

## Try the model

Now run the model to see that it behaves as expected.

First check the shape of the output:

In [28]:
for input_example_batch, target_example_batch in dataset.take(1):
    example_batch_predictions = model(input_example_batch)
    print(example_batch_predictions.shape, "# (batch_size, sequence_length, vocab_size)")

(64, 100, 48) # (batch_size, sequence_length, vocab_size)


In the above example the sequence length of the input is `100` but the model can be run on inputs of any length:

In [29]:
model.summary()

Model: "my_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       multiple                  12288     
                                                                 
 gru (GRU)                   multiple                  3938304   
                                                                 
 dense (Dense)               multiple                  49200     
                                                                 
Total params: 3999792 (15.26 MB)
Trainable params: 3999792 (15.26 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


To get actual predictions from the model you need to sample from the output distribution, to get actual character indices. This distribution is defined by the logits over the character vocabulary.

Note: It is important to _sample_ from this distribution as taking the _argmax_ of the distribution can easily get the model stuck in a loop.

Try it for the first example in the batch:

In [30]:
sampled_indices = tf.random.categorical(example_batch_predictions[0], num_samples=1)
sampled_indices = tf.squeeze(sampled_indices, axis=-1).numpy()

This gives us, at each timestep, a prediction of the next character index:

In [31]:
sampled_indices

array([33, 45,  7, 20, 13, 21, 24, 21, 29, 29, 19, 37, 34,  6, 33, 14,  3,
       27,  9, 13,  4, 41, 45, 46, 47,  9, 39, 44, 14, 28, 28,  4, 21,  8,
       45,  0, 42, 33, 28, 34,  6, 15,  7,  5, 44, 43, 47, 24, 46, 34, 46,
        9,  0, 30, 16, 23, 45, 15, 33, 13, 27,  5, 10, 42,  7, 33, 16, 13,
       23,  4, 37, 29, 21, 24, 13, 27, 13,  6, 32, 41, 38, 11, 28, 47, 36,
       36, 26, 46,  3,  6,  0,  2, 27, 13, 28, 14, 43, 38, 18, 21])

Decode these to see the text predicted by this untrained model:

In [32]:
print("Input:\n", text_from_ids(input_example_batch[0]).numpy())
print()
print("Next Char Predictions:\n", text_from_ids(sampled_indices).numpy())

Input:
 b'\xd8\xb2 \xd8\xaf\xd9\x88 \xd8\xb1\xd9\x88\xdb\x8c\xd9\x87 \xd9\x86\xd9\x87\xdb\x8c\xd8\xa8\n\xd9\x86\xdb\x8c\xd8\xa7\xd9\x85\xd8\xaf \xdb\x8c\xda\xa9\xdb\x8c \xd8\xb1\xd8\xa7 \xd8\xb3\xd8\xb1 \xd8\xa7\xd9\x86\xd8\xaf\xd8\xb1 \xd9\x86\xd8\xb4\xdb\x8c\xd8\xa8\n\xd8\xa8\xd8\xaf\xd9\x84 \xda\xaf\xd9\x81\xd8\xaa \xda\xa9\xd8\xa7\xd8\xb1\xdb\x8c \xd9\x86\xd9\x88 \xd8\xa2\xd9\x85\xd8\xaf \xd8\xa8\xd8\xb1\xd9\x88\xdb\x8c\n\xd9\x85\xd8\xb1\xd8\xa7 \xd8\xb2\xdb\x8c\xd9\x86 \xd8\xaf\xd9\x84\xdb\x8c\xd8\xb1\xd8\xa7\xd9\x86 \xd9\xbe\xd8\xb1\xd8\xae\xd8\xa7\xd8\xb4\xd8\xac\xd9\x88\xdb\x8c\n\xd9\x86\xd9\x87 \xd8\xa7\xd8\xb2 \xd8\xb4\xd9\x87\xd8\xb1 '

Next Char Predictions:
 b'\xd9\x81\xda\xaf\xd8\x8c\xd8\xae\xd8\xa6\xd8\xaf\xd8\xb2\xd8\xaf\xd8\xb7\xd8\xb7\xd8\xad\xd9\x86\xd9\x82\xc2\xbb\xd9\x81\xd8\xa7(\xd8\xb5\xd8\xa1\xd8\xa6)\xd9\xbe\xda\xaf\xdb\x8c\xe2\x80\x8c\xd8\xa1\xd9\x88\xda\xa9\xd8\xa7\xd8\xb6\xd8\xb6)\xd8\xaf\xd8\x9f\xda\xaf[UNK]\xda\x86\xd9\x81\xd8\xb6\xd9\x82\xc2\xbb\xd8\xa8\x

## Train the model

At this point the problem can be treated as a standard classification problem. Given the previous RNN state, and the input this time step, predict the class of the next character.

### Attach an optimizer, and a loss function

The standard `tf.keras.losses.sparse_categorical_crossentropy` loss function works in this case because it is applied across the last dimension of the predictions.

Because your model returns logits, you need to set the `from_logits` flag.


In [33]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=True)

In [34]:
example_batch_mean_loss = loss(target_example_batch, example_batch_predictions)
print("Prediction shape: ", example_batch_predictions.shape, " # (batch_size, sequence_length, vocab_size)")
print("Mean loss:        ", example_batch_mean_loss)

Prediction shape:  (64, 100, 48)  # (batch_size, sequence_length, vocab_size)
Mean loss:         tf.Tensor(3.8713303, shape=(), dtype=float32)


A newly initialized model shouldn't be too sure of itself, the output logits should all have similar magnitudes. To confirm this you can check that the exponential of the mean loss is approximately equal to the vocabulary size. A much higher loss means the model is sure of its wrong answers, and is badly initialized:

In [35]:
tf.exp(example_batch_mean_loss).numpy()

48.006203

Configure the training procedure using the `tf.keras.Model.compile` method. Use `tf.keras.optimizers.Adam` with default arguments and the loss function.

In [36]:
model.compile(optimizer='adam', loss=loss)

### Configure checkpoints

Use a `tf.keras.callbacks.ModelCheckpoint` to ensure that checkpoints are saved during training:

In [37]:
# Directory where the checkpoints will be saved
checkpoint_dir = './training_checkpoints'
# Name of the checkpoint files
checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}")

checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_prefix,
    save_weights_only=True)

### Execute the training

To keep training time reasonable, use 10 epochs to train the model. In Colab, set the runtime to GPU for faster training.

In [38]:
EPOCHS = 20

In [39]:
history = model.fit(dataset, epochs=EPOCHS, callbacks=[checkpoint_callback])

Epoch 1/20
395/395 [==============================] - 26s 52ms/step - loss: 2.2580
Epoch 2/20
395/395 [==============================] - 23s 53ms/step - loss: 1.5723
Epoch 3/20
395/395 [==============================] - 24s 56ms/step - loss: 1.3887
Epoch 4/20
395/395 [==============================] - 24s 58ms/step - loss: 1.3065
Epoch 5/20
395/395 [==============================] - 26s 61ms/step - loss: 1.2507
Epoch 6/20
395/395 [==============================] - 26s 61ms/step - loss: 1.2046
Epoch 7/20
395/395 [==============================] - 25s 60ms/step - loss: 1.1631
Epoch 8/20
395/395 [==============================] - 26s 61ms/step - loss: 1.1232
Epoch 9/20
395/395 [==============================] - 26s 61ms/step - loss: 1.0845
Epoch 10/20
395/395 [==============================] - 25s 60ms/step - loss: 1.0464
Epoch 11/20
395/395 [==============================] - 27s 61ms/step - loss: 1.0091
Epoch 12/20
395/395 [==============================] - 27s 61ms/step - loss: 0.9732
E

## Generate text

The simplest way to generate text with this model is to run it in a loop, and keep track of the model's internal state as you execute it.

![To generate text the model's output is fed back to the input](https://github.com/tensorflow/text/blob/master/docs/tutorials/images/text_generation_sampling.png?raw=1)

Each time you call the model you pass in some text and an internal state. The model returns a prediction for the next character and its new state. Pass the prediction and state back in to continue generating text.


The following makes a single step prediction:

In [40]:
class OneStep(tf.keras.Model):
  def __init__(self, model, chars_from_ids, ids_from_chars, temperature=1.0):
    super().__init__()
    self.temperature = temperature
    self.model = model
    self.chars_from_ids = chars_from_ids
    self.ids_from_chars = ids_from_chars

    # Create a mask to prevent "[UNK]" from being generated.
    skip_ids = self.ids_from_chars(['[UNK]'])[:, None]
    sparse_mask = tf.SparseTensor(
        # Put a -inf at each bad index.
        values=[-float('inf')]*len(skip_ids),
        indices=skip_ids,
        # Match the shape to the vocabulary
        dense_shape=[len(ids_from_chars.get_vocabulary())])
    self.prediction_mask = tf.sparse.to_dense(sparse_mask)

  @tf.function
  def generate_one_step(self, inputs, states=None):
    # Convert strings to token IDs.
    input_chars = tf.strings.unicode_split(inputs, 'UTF-8')
    input_ids = self.ids_from_chars(input_chars).to_tensor()

    # Run the model.
    # predicted_logits.shape is [batch, char, next_char_logits]
    predicted_logits, states = self.model(inputs=input_ids, states=states,
                                          return_state=True)
    # Only use the last prediction.
    predicted_logits = predicted_logits[:, -1, :]
    predicted_logits = predicted_logits/self.temperature
    # Apply the prediction mask: prevent "[UNK]" from being generated.
    predicted_logits = predicted_logits + self.prediction_mask

    # Sample the output logits to generate token IDs.
    predicted_ids = tf.random.categorical(predicted_logits, num_samples=1)
    predicted_ids = tf.squeeze(predicted_ids, axis=-1)

    # Convert from token ids to characters
    predicted_chars = self.chars_from_ids(predicted_ids)

    # Return the characters and model state.
    return predicted_chars, states

In [41]:
one_step_model = OneStep(model, chars_from_ids, ids_from_chars)

Run it in a loop to generate some text. Looking at the generated text, you'll see the model knows when to capitalize, make paragraphs and imitates a Shakespeare-like writing vocabulary. With the small number of training epochs, it has not yet learned to form coherent sentences.

In [51]:
start = time.time()
states = None
next_char = tf.constant(['کزین'])
result = [next_char]

for n in range(1000):
  next_char, states = one_step_model.generate_one_step(next_char, states=states)
  result.append(next_char)

result = tf.strings.join(result)
end = time.time()
print(result[0].numpy().decode('utf-8'), '\n\n' + '_'*80)
print('\nRun time:', end - start)

کزین سخن نگذرم
وگر با نژاد آید این رای‌زن
همی گفت با من که با توجهان
چوآشان به از رنج گفتن نداشت
زبان را یمانی یکی کار زیر
ندارم که دارند ما را ز اندوه و داد
که گیتی همی گوید از راه باز
نیارست گفت ایچ ازین پس هوا
ندارد کسی سبز بازی مرد
چو آگه شد از شاه پرده‌سرای
بزرگان روشن‌دل و شیر کیست
به رزم اندر آیین نیکوترست
کجا آن همه رفتمی آشتید
ازو دور شد خورد و آرام و ناز
جهانی به آرایش چین نهاد
همان نیز ناکام و گردنکشان
زبان کس کنم جان تو کشورت
بر آسود اگر زنده پالیزبان
ازیشان کسی را نداند نه بر
بر آشوبد این نیکویها درخت
ازان ننگ بر نای شاهی کنید
وگر نامداری و چه آفرین
کنون چون بپوشد کمند انگیر
از ایشان نکردند یاد آر شیر
گر از دشمن جز بداد و به شاه
فرستد به نزدیک شاه اندرند
یکی مرگ خویش از برش تنگ دور
همی سازد از بار شاخی بمرد
فراوان ز تنبر بلند آسمان
دل و دیده گشتند و پر غم به خاک
نکرد اشتر و نالهٔ بر سرش
همه بهمن از داد یکسر نهید
پسر را به نیکی ترا نیست راه
هم از تو من آید به فرمان تو راست
زمانم فگندن به دشوارتر
پسر پادشاهی که از کین نبود
اگرچه ز دادار کار آگهان
همه شب همی خیره خاک تنگ
بهار

The easiest thing you can do to improve the results is to train it for longer (try `EPOCHS = 30`).

You can also experiment with a different start string, try adding another RNN layer to improve the model's accuracy, or adjust the temperature parameter to generate more or less random predictions.

If you want the model to generate text *faster* the easiest thing you can do is batch the text generation. In the example below the model generates 5 outputs in about the same time it took to generate 1 above.

## Export the generator

This single-step model can easily be [saved and restored](https://www.tensorflow.org/guide/saved_model), allowing you to use it anywhere a `tf.saved_model` is accepted.

In [52]:
tf.saved_model.save(one_step_model, 'one_step')
one_step_reloaded = tf.saved_model.load('one_step')

In [45]:
states = None
next_char = tf.constant(['ROMEO:'])
result = [next_char]

for n in range(100):
  next_char, states = one_step_reloaded.generate_one_step(next_char, states=states)
  result.append(next_char)

print(tf.strings.join(result)[0].numpy().decode("utf-8"))

ROMEO:م فرو بردبار
بدین بوم ویران بود گر کمان
چوفرزند باشد به پرهیز این
چو بر گردگذران بنالد همی
گر از نام
